In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Putting it together: the landscape of reinforcement learning methods

## Where this session fits

> **The question we start from: how do all these methods fit together, and how do we choose among them for a new problem?**

- **Position: the closing session, no new method**
    - Part A built the value-based path (Bellman and dynamic programming, sampling, control, deep Q-learning)
    - Part B built the policy-gradient methods (the theorem, the baseline and the advantage, the critic, how far)
    - Part C applied them to language models: PPO on a language model, and GRPO
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. place any method of the series by three questions: is a model needed, what does it learn, which policy generates its data
2. read the threads of the series (bias and variance, backup width and depth, on- and off-policy, evaluate versus improve, exploration) across all three parts
3. say which equation carries each idea of the series
4. choose a method family for a new problem, and say what the series left out


# Three questions place every method

**Why this section?** There are many methods. Three questions give each one an address.

- **Is a model needed?**
    - *model-based:* a model of the transitions and rewards is used for planning (known here: value and policy iteration)
    - *model-free:* only sampled experience (everything from Monte Carlo on)
- **What does it learn?**
    - *values:* $\statevalfun$ or $\actvalfun$
    - the policy is read off (the greedy action)
    - *a policy:* $\pi_\theta$ directly, by gradient ascent (Part B)
    - *both:* actor-critic methods (actor-critic, PPO) learn a policy and a value (the critic)
- **Which policy generates the data?**
    - *on-policy:* the policy being learned (SARSA, REINFORCE)
    - PPO and GRPO are *near* on-policy (a slightly older copy, corrected by the ratio)
    - *off-policy:* another policy, or fixed data (Q-learning, DQN)


**The grid**

<img src="images/methods_grid.png" width=80%>

- the rows combine two questions: with a known model there is no sampled data, so the model-based row has no on-/off-policy split
- the value-based path (green and blue) sits in the left column
- Part B (orange) in the middle and right
- Part C's PPO and GRPO are in the same cells as their Part B relatives
- two kinds of empty cell
    - *none here*: no model-based policy method in this series
    - *not covered*: off-policy actor-critic methods exist, but the series skips them
- the three questions are listed in the order values, both, policy in the columns


## Table 1, extended: all the methods

**Why this section?** Table 1 (the value-based methods, end of Part A) condensed, then the Part B and Part C rows.

**Terms from earlier sessions**

- *p*-step TD: look $p$ steps ahead
- GAE (generalized advantage estimation): a blend of depths
- critic: the learned value used by an actor
- ratio: new over old probability of an action

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Model?</div></th><th nowrap><div align="left">Learns</div></th><th nowrap><div align="left">Data from</div></th><th nowrap><div align="left">Depth</div></th><th nowrap><div align="left">Bias of the weight or target</div></th><th nowrap><div align="left">Variance</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">dynamic programming</div></td><td nowrap><div align="left">yes</div></td><td nowrap><div align="left">values</div></td><td nowrap><div align="left">none (planning)</div></td><td nowrap><div align="left">1 step, all successors</div></td><td nowrap><div align="left">uses the current estimate</div></td><td nowrap><div align="left">none from sampling</div></td></tr>
<tr><td nowrap><div align="left">Monte Carlo</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">values</div></td><td nowrap><div align="left">policy evaluated</div></td><td nowrap><div align="left">to the end</div></td><td nowrap><div align="left">none</div></td><td nowrap><div align="left">high</div></td></tr>
<tr><td nowrap><div align="left">TD / Q-learning / SARSA / DQN</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">values</div></td><td nowrap><div align="left">on-policy (SARSA) or off-policy</div></td><td nowrap><div align="left">1 step ($p$-step between)</div></td><td nowrap><div align="left">while the estimate is wrong (Q-learning and DQN add <i>maximization bias</i>, from the max)</div></td><td nowrap><div align="left">lower</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy</div></td><td nowrap><div align="left">on-policy</div></td><td nowrap><div align="left">to the end</div></td><td nowrap><div align="left">none</div></td><td nowrap><div align="left">high (lower with a baseline, still high)</div></td></tr>
<tr><td nowrap><div align="left">actor-critic</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy and value</div></td><td nowrap><div align="left">on-policy</div></td><td nowrap><div align="left">1 step ($p$-step or GAE between)</div></td><td nowrap><div align="left">from the critic</div></td><td nowrap><div align="left">lower</div></td></tr>
<tr><td nowrap><div align="left">PPO</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy and value</div></td><td nowrap><div align="left">near on-policy (ratio, clip)</div></td><td nowrap><div align="left">GAE</div></td><td nowrap><div align="left">from the critic (clipping limits the step: slot 2, not the weight)</div></td><td nowrap><div align="left">lower</div></td></tr>
<tr><td nowrap><div align="left">GRPO</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy</div></td><td nowrap><div align="left">near on-policy (group per prompt)</div></td><td nowrap><div align="left">whole response</div></td><td nowrap><div align="left">a shrink by (G-1)/G and a rescaling by the standard deviation</div></td><td nowrap><div align="left">noisier baseline, no critic</div></td></tr>
<tr><td nowrap><div align="left">RLHF</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy (reward from a reward model)</div></td><td nowrap><div align="left">on-policy samples</div></td><td nowrap><div align="left">as PPO</div></td><td nowrap><div align="left">the reward model is a proxy</div></td><td nowrap><div align="left">as PPO</div></td></tr>
</tbody></table>

- the **depth** and **bias** columns carry over from Part A to the weight of the policy gradient: the same dial


## The slot table, in the grid

**Why this section?** The methods of Part B, and GRPO of Part C, are one theorem. The rows differ only in three choices.

| Method | Slot 1: weight | Slot 2: how far | Slot 3: baseline source |
|:--|:--|:--|:--|
| raw theorem | return | plain step | none |
| REINFORCE | return minus baseline | plain step | moving average |
| actor-critic | TD error or GAE | plain step | learned critic |
| PPO | GAE with a critic | clipped ratio, several epochs | learned critic |
| GRPO | group-relative advantage | clipped ratio, several epochs | group mean |

- in the grid, all these rows are in the middle and right columns, near on-policy
- RLHF-style recipes that use PPO or GRPO make choices from this table
    - they add a reward source (program, reward model, judge)
    - and usually a KL penalty to a reference model


## The threads, one last time

**Why this section?** Each thread appeared early and came back. This is the whole path of each.

> **A thread is a theme that returns in several sessions, each time in a new form. Read the arrow as "comes back as", in session order.**

<table>
<thead><tr><th nowrap><div align="left">Thread</div></th><th nowrap><div align="left">The question it keeps asking</div></th><th nowrap><div align="left">Its path, from the first session to the last</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left"><b>bias and variance</b></div></td><td nowrap><div align="left">how noisy, and how biased, is the number we learn from?</div></td><td nowrap><div align="left">the targets of Monte Carlo and TD $\to$ the weight in the policy gradient $\to$ baseline, critic, group mean (clipping, separately, limits the step)</div></td></tr>
<tr><td nowrap><div align="left"><b>backup width and depth</b></div></td><td nowrap><div align="left">how many steps ahead, and how many branches, does one update use?</div></td><td nowrap><div align="left">dynamic programming, Monte Carlo, $p$-step TD $\to$ return-to-go, TD error, GAE</div></td></tr>
<tr><td nowrap><div align="left"><b>on-policy versus off-policy</b></div></td><td nowrap><div align="left">whose samples does the update learn from?</div></td><td nowrap><div align="left">light at Monte Carlo, defined at SARSA versus Q-learning $\to$ replay allowed in DQN $\to$ the ratio in PPO</div></td></tr>
<tr><td nowrap><div align="left"><b>evaluate versus improve</b></div></td><td nowrap><div align="left">how do we alternate judging a policy and changing it?</div></td><td nowrap><div align="left">policy iteration $\to$ control with values $\to$ the critic and the actor</div></td></tr>
<tr><td nowrap><div align="left"><b>exploration</b></div></td><td nowrap><div align="left">how does the agent try things it has not tried?</div></td><td nowrap><div align="left">greedy versus $\epsilon$-greedy $\to$ a stochastic policy that explores by sampling $\to$ a group that must differ</div></td></tr>
<tr><td nowrap><div align="left"><b>the proxy problem</b> (new in Part C)</div></td><td nowrap><div align="left">does the reward measure what we want?</div></td><td nowrap><div align="left">only in Part C: the reward is a stand-in, so it can be exploited (reward hacking); the KL penalty to a reference model limits how far</div></td></tr>
</tbody></table>


# Equations behind the ideas (1): value-based

**Why this section?** Each idea of Part A condensed to its equation.

<table>
<thead><tr><th nowrap><div align="left">Idea</div></th><th nowrap><div align="left">The equation</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">the return</div></td><td nowrap><div align="left">$\rlret_\tt = \rewseq_{\tt+1} + \disc \, \rlret_{\tt+1}$</div></td></tr>
<tr><td nowrap><div align="left">the Bellman equation</div></td><td nowrap><div align="left">$\statevalfun_\pi(\state) = \sum_\act \pi(\act \mid \state) \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act) \bigl( \rew + \disc \, \statevalfun_\pi(\state') \bigr)$</div></td></tr>
<tr><td nowrap><div align="left">TD learning</div></td><td nowrap><div align="left">$\statevalfun(\stateseq_\tt) \leftarrow \statevalfun(\stateseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \statevalfun(\stateseq_{\tt+1}) - \statevalfun(\stateseq_\tt) \bigr]$</div></td></tr>
<tr><td nowrap><div align="left">SARSA</div></td><td nowrap><div align="left">$\actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \actvalfun(\stateseq_{\tt+1}, \actseq_{\tt+1}) - \actvalfun(\stateseq_\tt, \actseq_\tt) \bigr]$</div></td></tr>
<tr><td nowrap><div align="left">Q-learning</div></td><td nowrap><div align="left">$\actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun(\stateseq_{\tt+1}, \act') - \actvalfun(\stateseq_\tt, \actseq_\tt) \bigr]$</div></td></tr>
</tbody></table>


# Equations behind the ideas (2): policy-based and language models

**Why this section?** Part B and Part C in the same form. The advantage and the ratio are defined in their sessions. Here they are named.

<table>
<thead><tr><th nowrap><div align="left">Idea</div></th><th nowrap><div align="left">The equation</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">the policy gradient theorem (return-to-go variant)</div></td><td nowrap><div align="left">$\nabla_\theta J(\theta) = \Exp{\tau \sim \pi_\theta} \bigl[ \sum_\tt \nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, \rlret_{\tt,\tau} \bigr]$</div></td></tr>
<tr><td nowrap><div align="left">the advantage form</div></td><td nowrap><div align="left">the same with the weight $\advseq_\tt$ in place of $\rlret_{\tt,\tau}$: the return minus a baseline, or an estimate such as the TD error $\delta_\tt = \rewseq_{\tt+1} + \disc \, \statevalfun(\stateseq_{\tt+1}) - \statevalfun(\stateseq_\tt)$ or GAE</div></td></tr>
<tr><td nowrap><div align="left">the probability ratio</div></td><td nowrap><div align="left">$r_\tt(\theta) = \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, / \, \pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)$</div></td></tr>
<tr><td nowrap><div align="left">PPO (clipped objective)</div></td><td nowrap><div align="left">$\min{} \bigl( r_\tt \, \hat{\advseq}_\tt, \; \text{clip}(r_\tt, \, 1 - \epsilon_{\text{clip}}, \, 1 + \epsilon_{\text{clip}}) \, \hat{\advseq}_\tt \bigr)$, with $\hat{\advseq}_\tt$ an <i>estimate</i> of the advantage ($\epsilon_{\text{clip}}$ is not the $\epsilon$ of $\epsilon$-greedy)</div></td></tr>
<tr><td nowrap><div align="left">GRPO advantage</div></td><td nowrap><div align="left">$\hat{\advseq}_i = (\rlret_i - \text{mean}) / \text{std}$ over a group of completions, in the same clipped objective</div></td></tr>
<tr><td nowrap><div align="left">KL penalty (language models)</div></td><td nowrap><div align="left">subtract $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$, $\beta$ the weight, $\pi_{\text{ref}}$ a frozen reference model: a per-token reward in PPO, a loss term in GRPO</div></td></tr>
</tbody></table>


## Choosing a family

**Why this section?** From knowing the methods to choosing among them for a new problem.

- **the model is known and small** (a board game's rules, a grid): dynamic programming
- **small discrete states and actions, model unknown:** tabular Q-learning or SARSA
- **large or continuous states, a few discrete actions:** DQN
- **continuous actions, or a stochastic policy is needed:** policy-gradient methods
    - PPO is a common on-policy default (off-policy methods such as SAC (soft actor-critic) are often used for continuous control and are not covered here)
- **language models:** the policy is the model
    - the reward comes from a program, a reward model or a judge
    - PPO and GRPO are the tools
- **only fixed data, no environment:** offline methods (left out of this series)
- **samples are costly or unsafe:** learn or use a model, or offline data (not covered)
- the grid's three questions decide
    - do we have a model?
    - what must we learn (values or a policy)?
    - can the data be reused (on- or off-policy)?
- in every case
    - reward design decides what the agent learns (reward hacking)
    - exploration decides what *can* be learned (a common view: RL mostly raises the probability of what the policy already tries)


## What this series left out

**Why this section?** A map should show its edges.

- **learning a model** of the environment and planning with it (model-based RL with a learned model)
- **off-policy actor-critic methods** for continuous actions (the empty cell in the grid)
- **continuous-action value methods and distributional or ensemble variants** of DQN
- **the theory** behind convergence of these methods, beyond the tabular case
- **offline RL**
- **natural-gradient methods**: updates that follow the geometry of the policy distribution rather than raw parameters
    - and trust-region theory beyond the short TRPO mention
- **maximum-entropy methods**, **importance-sampling corrections** for multi-step off-policy learning
- **imitation learning** and **hierarchical RL**
- **multi-agent and partially observed problems**, beyond what the examples showed
- **the language-model details** (reward-model training at scale, RLHF at scale, GRPO variants, the derivation and use of DPO): the Advanced course


# What we established, and what it sets up

- **Established**
    - one loop, from Frozen Lake and Microduck to language models
    - three questions place every method: model, what is learned, which policy generates the data
    - the same few threads run through both the value-based and the policy-gradient methods
        - bias and variance, backup depth, on- and off-policy, evaluate versus improve, exploration
    - Part C added the proxy problem
    - the equations behind the main ideas fit on two slides
- **Sets up**
    - the Advanced course: reward models, RLHF, GRPO with verifiable rewards, preference optimization
    - the code notebooks and lab exercises that implement these methods
